In [122]:
from langchain_chroma import Chroma
from pyprojroot import here
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from pprint import  pprint
from dotenv import load_dotenv

load_dotenv()

True

In [123]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
VECTORDB_DIR = "data/airline_policy_vectordb"
DOC_DIR="data/unstructured_docs"
K=2


In [124]:


class PolicyAnswer(BaseModel):
    found: bool = Field(
        description="true if the answer exists in the provided policy content"
    )
    answer: str = Field(
        description="answer extracted from the content or 'The answer does not exist.'"
    )


In [125]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)
vectordb = Chroma(
    persist_directory=str(here(VECTORDB_DIR)),
    embedding_function=embeddings,
    collection_name="rag-chroma",
)

print("Number of vectors in vectordb:", vectordb._collection.count(), "\n\n")

Number of vectors in vectordb: 22 




In [126]:
message = "What is the cancelation rule for a flight ticket at swiss airline policy?"

In [127]:
docs = vectordb.similarity_search(message, k=K)

In [128]:
docs

[Document(id='61744360-977f-4f13-826c-ad6e073af790', metadata={'moddate': '2024-09-08T21:34:59+00:00', 'creationdate': '2024-09-08T21:34:59+00:00', 'source': 'D:\\WorkStation\\Python\\Agentic-AI\\RAG_LLM_SQL\\data\\unstructured_docs\\swiss_airline_policy\\swiss_faq.pdf', 'total_pages': 12, 'title': 'Merged with PDFCreator Online', 'creator': 'PDFCreator Online (www.pdfforge.org/online)', 'page_label': '10', 'producer': 'PDFCreator Online (www.pdfforge.org/online)', 'page': 9}, page_content="for a refund or may only be able to receive a partial refund. If you booked your flight through a third-party website or\ntravel agent, you may need to contact them directly to cancel your flight. Always check the terms and conditions of your\nticket to make sure you understand the cancellation policy and any associated fees or penalties. If you're cancelling your\nflight due to unforeseen circumstances such as a medical emergency or a natural disaster, Swiss Air may offer you\nspecial exemptions or

In [129]:
question = "# User new question:\n" + message
retrieved_content = ""
for doc in docs:
    retrieved_content += f"{doc.page_content}\n\n"
prompt = f"# Content:\n{retrieved_content}\n\n{question}"

In [130]:
pprint(prompt)

('# Content:\n'
 'for a refund or may only be able to receive a partial refund. If you booked '
 'your flight through a third-party website or\n'
 'travel agent, you may need to contact them directly to cancel your flight. '
 'Always check the terms and conditions of your\n'
 'ticket to make sure you understand the cancellation policy and any '
 "associated fees or penalties. If you're cancelling your\n"
 'flight due to unforeseen circumstances such as a medical emergency or a '
 'natural disaster, Swiss Air may offer you\n'
 'special exemptions or accommodations. What is Swiss Airlines 24 Hour '
 'Cancellation Policy? Swiss Airlines has a 24\n'
 '\n'
 'How to Cancel a Swiss Air Flight: 877-\n'
 '5O7-7341 Step-by-Step Guide\n'
 'Swiss Air is a premium airline based in Switzerland that offers a range of '
 'domestic and international flights to\n'
 'passengers. However, sometimes situations arise where passengers may need to '
 'cancel their flights. In such cases, it is\n'
 'important 

In [ ]:
llm = HuggingFaceEndpoint(
    model = "google/gemma-2-2b-it",
    task="text-generation"
)

model = ChatHuggingFace(llm = llm)

In [132]:
chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a Swiss Airlines policy assistant.\n"
        "Rules:\n"
        "- Answer ONLY from the provided policy content\n"
        "- Do NOT use outside knowledge\n"
        "- If the answer is not explicitly present, set found=false\n"
        "- If found=false, answer MUST be exactly: 'The answer does not exist.'\n\n"
        "{format_instructions}"
    ),
    (
        "human",
        "Policy Content:\n{context}\n\nUser Question:\n{question}"
    )
])

In [133]:
parser = PydanticOutputParser(pydantic_object=PolicyAnswer)
format_instructions = parser.get_format_instructions()

In [134]:
chain = chat_prompt | model | parser

In [135]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

query = "What is the cancellation rule for a flight ticket at Swiss Airlines?"

docs = vectordb.similarity_search(query, k=K)

result = chain.invoke({
    "context": format_docs(docs),
    "question": query,
    "format_instructions": format_instructions
})

from pprint import pprint
pprint(result)


PolicyAnswer(found=True, answer="Swiss Airlines has a 24 hour cancellation policy that allows passengers to cancel their flights within 24 hours of booking without penalty. If you booked your flight through a travel agent or third-party website, you’ll need to check their cancellation policy. If you cancel your Swiss Airlines flight after the 24 hour window, you may be subject to cancellation fees or penalties.  If you have a non-refundable ticket and cancel your flight within 24 hours of booking, you'll receive a full refund of your ticket price. However, if you cancel your flight after the 24 hour window, you may not be eligible for a refund. ")


In [136]:
from langchain_core.tools import tool

@tool
def lookup_swiss_airline_policy(query: str) -> str:
    """Search within the Swiss Airline's company policies to check whether certain options are permitted. Input should be a search query."""
    vectordb = Chroma(
        collection_name="rag-chroma",
        persist_directory=str(here(VECTORDB_DIR)),
        embedding_function=embeddings,
    )
    docs = vectordb.similarity_search(query, k=K)
    return "\n\n".join([doc.page_content for doc in docs])

In [137]:
print(lookup_swiss_airline_policy.name)
print(lookup_swiss_airline_policy.args)
print(lookup_swiss_airline_policy.description)

lookup_swiss_airline_policy
{'query': {'title': 'Query', 'type': 'string'}}
Search within the Swiss Airline's company policies to check whether certain options are permitted. Input should be a search query.


In [138]:
pprint(lookup_swiss_airline_policy.invoke("can I cancel my ticket?"))

('for a refund or may only be able to receive a partial refund. If you booked '
 'your flight through a third-party website or\n'
 'travel agent, you may need to contact them directly to cancel your flight. '
 'Always check the terms and conditions of your\n'
 'ticket to make sure you understand the cancellation policy and any '
 "associated fees or penalties. If you're cancelling your\n"
 'flight due to unforeseen circumstances such as a medical emergency or a '
 'natural disaster, Swiss Air may offer you\n'
 'special exemptions or accommodations. What is Swiss Airlines 24 Hour '
 'Cancellation Policy? Swiss Airlines has a 24\n'
 '\n'
 'hour cancellation policy that allows passengers to cancel their flights '
 'within 24 hours of booking at +1-877-507-7341\n'
 'without penalty. This policy applies to all fare types, including '
 'non-refundable tickets. If you cancel your Swiss Airlines\n'
 "flight within 24 hours of booking, you'll receive a full refund of your "
 'ticket price.\n'
 